In [20]:
from anomalouslib.models.model_handler import save_model, load_model, get_class_def_from_notebook
from anomalouslib.models.testing_utils import evaluate_continuous, predict_continuous
from anomalouslib.data.preprocessing import generate_classification_tensors
from anomalouslib.data.DatasetGenerator import DatasetGenerator
from anomalouslib.data.preprocessing import min_max_normalize
from anomalouslib.models.training_utils import train_model
from sklearn.preprocessing import LabelEncoder
from anomalouslib.constants import *

from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
import pandas as pd
import torch
import numpy as np

In [2]:
# Aconseguim el dataset
num_particules = 10000
dg = DatasetGenerator(num_particules)
data = dg.generate_FractionalBrownianMotion(random_params=True, num_passos=200)

c:\Users\jmass\Documents\GitHub\TFG GEINF\AnomalousApp\venv\Lib\site-packages\fbm\fbm.py:171: UserWarning: Combination of increments n and Hurst value H invalid for Davies-Harte method. Reverting to Hosking method. Occurs when n is small and Hurst is close to 1. 
  warnings.warn(


In [3]:
# Aconseguim un dataframe amb les coordenades i la hurst
df = data[["coords", "attributes"]].copy()
df["hurst"] = df["attributes"].apply(lambda x: x["hurst"])
df = df[["coords", "hurst"]]

In [4]:
# Tractem les coordenades per aconseguir X_tensor
coords_list = [np.stack([row[0], row[1]], axis=1) for row in df['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1) # Per aconseguir la forma (n_samples, 2, seq_length)

In [7]:
# Tractem les coordenades per aconseguir y_tensor
hurst_series = df["hurst"].apply(lambda x: min_max_normalize(x, params.fbm.min_hurst, params.fbm.max_hurst))
labels = hurst_series.values  # Suponiendo que 'hurst' contiene los valores reales
y_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

In [8]:
# Aconseguir X_train, X_val, X_test, y_train, y_val i y_test
n, train_end, val_end = len(df), int(0.7 * len(df)), int(0.85 * len(df))
X_train, X_val, X_test = X_tensor[:train_end], X_tensor[train_end:val_end], X_tensor[val_end:]
y_train, y_val, y_test = y_tensor[:train_end], y_tensor[train_end:val_end], y_tensor[val_end:]

In [9]:
# Muntem els Data_loaders
learning_batch_size = 32
try_batch_size = 1

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=learning_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=try_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=try_batch_size, shuffle=True)

In [10]:
class FractionalBrownianMotionHurstRegressor(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        # Bloque 1
        self.first_conv = nn.Conv1d(input_channels, 32, 3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, 3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, 3, padding=1)
        
        # Bloque 2
        self.conv_32_64 = nn.Conv1d(32, 64, 3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, 3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, 3, padding=1)
        
        # Bloque 3
        self.conv_64_128 = nn.Conv1d(64, 128, 3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, 3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, 3, padding=1)
        
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool1d(4)  # Corregido a 1D
        self.fc1 = nn.Linear(128*4, 256)         # Ajustada la entrada
        self.fc2 = nn.Linear(256, 1)
        
        # Opcional: Añadir BatchNorm
        self.bn32 = nn.BatchNorm1d(32)
        self.bn64 = nn.BatchNorm1d(64)
        self.bn128 = nn.BatchNorm1d(128)

        self.dropout_conv = nn.Dropout(p=0.1)   # Després de blocs convolucionals
        self.dropout_fc = nn.Dropout(p=0.5)     # Abans de la capa de sortida

    def forward(self, x):
        # Bloque 1
        a_resnet = F.relu(self.first_conv(x))
        a = F.relu(self.conv1_32(a_resnet))
        a = F.relu(self.conv2_32(a))
        a = a + a_resnet
        a = self.dropout_conv(a)  # Dropout després del bloc residual
        a = self.max_pool(a)
        # Bloque 2
        b_resnet = F.relu(self.conv_32_64(a))
        b = F.relu(self.conv1_64(b_resnet))
        b = F.relu(self.conv2_64(b))
        b = b + b_resnet
        b = self.dropout_conv(b)
        b = self.max_pool(b)

        # Bloque 3
        c_resnet = F.relu(self.conv_64_128(b))
        c = F.relu(self.conv1_128(c_resnet))
        c = F.relu(self.conv2_128(c))
        c = c + c_resnet
        c = self.dropout_conv(c)
        c = self.avg_pool(c)

        # Clasificación
        c = c.view(c.size(0), -1)
        c = F.relu(self.fc1(c))
        c = self.dropout_fc(c)  # Dropout abans de la capa de sortida
        c = self.fc2(c)
        c = torch.tanh(c)
        out = (c + 1) / 2
        
        return out

input_channels = 2

model_params = {
    "input_channels": input_channels
}

model = FractionalBrownianMotionHurstRegressor(**model_params)


In [11]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3),
    mode='min',
    factor=0.1,
    patience=2,
    # verbose=True <--- Deprecated
)

best_model = train_model(
    model=model,
    criterion=nn.MSELoss(),
    optimizer=scheduler.optimizer,
    num_epochs=15,
    train_loader=train_loader,
    val_loader=val_loader,
    scheduler=scheduler,
    print_epoch=True,
    patience=5
)

Epoch 1/15
Train MSE Loss: 0.0427 | Val MSE Loss: 0.0176
Epoch 2/15
Train MSE Loss: 0.0110 | Val MSE Loss: 0.0187
Epoch 3/15
Train MSE Loss: 0.0057 | Val MSE Loss: 0.0191
Epoch 4/15
Train MSE Loss: 0.0050 | Val MSE Loss: 0.0126
Epoch 5/15
Train MSE Loss: 0.0043 | Val MSE Loss: 0.0215
Epoch 6/15
Train MSE Loss: 0.0038 | Val MSE Loss: 0.0187
Epoch 7/15
Train MSE Loss: 0.0038 | Val MSE Loss: 0.0192
Epoch 8/15
Train MSE Loss: 0.0030 | Val MSE Loss: 0.0203
Epoch 9/15
Train MSE Loss: 0.0029 | Val MSE Loss: 0.0208
Early stopping!


In [12]:
evaluate_continuous(best_model, X_test, y_test)

Test MSE: 0.0130


0.012989167124032974

In [13]:
def predict_continuous(model, X_input, y_input=None, num_samples=None, to_print=True):
    if num_samples is not None:
        X_input = X_input[:num_samples]
        if y_input is not None:
            y_input = y_input[:num_samples]

    model.eval()
    with torch.no_grad():
        output = model(X_input)
        predicted_value = output.cpu().numpy()

    results = {'Prediction': predicted_value.flatten()}
    if y_input is not None:
        true_values = y_input.cpu().numpy().flatten()
        results['Real Value'] = true_values

    df = pd.DataFrame(results)

    if to_print:
        print("Predicciones:")
        for i, row in df.iterrows():
            if 'Real Value' in df.columns:
                print(f"Ejemplo {i+1}: Predicho = {row['Prediction']}, Real = {row['Real Value']}")
            else:
                print(f"Ejemplo {i+1}: Predicho = {row['Prediction']}")

    return df

In [14]:
predict_continuous(best_model, X_test, y_test, num_samples=10)

Predicciones:
Ejemplo 1: Predicho = 0.8721164464950562, Real = 0.7396455407142639
Ejemplo 2: Predicho = 0.13517439365386963, Real = 0.10985244810581207
Ejemplo 3: Predicho = 0.7813926935195923, Real = 0.5924985408782959
Ejemplo 4: Predicho = 0.34026914834976196, Real = 0.2926650643348694
Ejemplo 5: Predicho = 0.2552390694618225, Real = 0.21963447332382202
Ejemplo 6: Predicho = 0.7228571176528931, Real = 0.5153778195381165
Ejemplo 7: Predicho = 0.4252484440803528, Real = 0.3774055242538452
Ejemplo 8: Predicho = 0.8827221393585205, Real = 0.9563958048820496
Ejemplo 9: Predicho = 0.8854444026947021, Real = 0.8862900137901306
Ejemplo 10: Predicho = 0.8374061584472656, Real = 0.7645677328109741


,Prediction,Real Value
0,0.872116,0.739646
1,0.135174,0.109852
2,0.781393,0.592499
3,0.340269,0.292665
4,0.255239,0.219634
5,0.722857,0.515378
6,0.425248,0.377406
7,0.882722,0.956396
8,0.885444,0.886290
9,0.837406,0.764568


In [16]:
SAVE_PATH = "../../../models/trained_models/"

In [17]:
save_model(model_instance=best_model,
           model_type="regressor",
           model_source=get_class_def_from_notebook(best_model.__class__.__name__),
           params=model_params,
           input_channels=input_channels,
           path="."
           )

In [25]:
prova_carrega_model=load_model(
    best_model.__class__.__name__ +
    ".pth"
)

evaluate_continuous(prova_carrega_model, X_test, y_test)
predict_continuous(prova_carrega_model, X_test, y_test, num_samples=10)

Test MSE: 0.0130
Predicciones:
Ejemplo 1: Predicho = 0.8721164464950562, Real = 0.7396455407142639
Ejemplo 2: Predicho = 0.13517439365386963, Real = 0.10985244810581207
Ejemplo 3: Predicho = 0.7813926935195923, Real = 0.5924985408782959
Ejemplo 4: Predicho = 0.34026914834976196, Real = 0.2926650643348694
Ejemplo 5: Predicho = 0.2552390694618225, Real = 0.21963447332382202
Ejemplo 6: Predicho = 0.7228571176528931, Real = 0.5153778195381165
Ejemplo 7: Predicho = 0.4252484440803528, Real = 0.3774055242538452
Ejemplo 8: Predicho = 0.8827221393585205, Real = 0.9563958048820496
Ejemplo 9: Predicho = 0.8854444026947021, Real = 0.8862900137901306
Ejemplo 10: Predicho = 0.8374061584472656, Real = 0.7645677328109741


,Prediction,Real Value
0,0.872116,0.739646
1,0.135174,0.109852
2,0.781393,0.592499
3,0.340269,0.292665
4,0.255239,0.219634
5,0.722857,0.515378
6,0.425248,0.377406
7,0.882722,0.956396
8,0.885444,0.886290
9,0.837406,0.764568
